In [182]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [183]:
df = pd.read_csv('D:\\Player salary\\Players.csv')

In [184]:
df.head()

,Unnamed: 0,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,...,CRuns,CRBI,CWalks,League,Division,PutOuts,Assists,Errors,Salary,NewLeague
0,-Andy Allanson,293,66,1,30,29,14,1,293,66,...,30,29,14,A,E,446,33,20,NaN,A
1,-Alan Ashby,315,81,7,24,38,39,14,3449,835,...,321,414,375,N,W,632,43,10,475.0,N
2,-Alvin Davis,479,130,18,66,72,76,3,1624,457,...,224,266,263,A,W,880,82,14,480.0,A
3,-Andre Dawson,496,141,20,65,78,37,11,5628,1575,...,828,838,354,N,E,200,11,3,500.0,N
4,-Andres Galarraga,321,87,10,39,42,30,2,396,101,...,48,46,33,N,E,805,40,4,91.5,N


In [185]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  322 non-null    object 
 1   AtBat       322 non-null    int64  
 2   Hits        322 non-null    int64  
 3   HmRun       322 non-null    int64  
 4   Runs        322 non-null    int64  
 5   RBI         322 non-null    int64  
 6   Walks       322 non-null    int64  
 7   Years       322 non-null    int64  
 8   CAtBat      322 non-null    int64  
 9   CHits       322 non-null    int64  
 10  CHmRun      322 non-null    int64  
 11  CRuns       322 non-null    int64  
 12  CRBI        322 non-null    int64  
 13  CWalks      322 non-null    int64  
 14  League      322 non-null    object 
 15  Division    322 non-null    object 
 16  PutOuts     322 non-null    int64  
 17  Assists     322 non-null    int64  
 18  Errors      322 non-null    int64  
 19  Salary      263 non-null    f

In [186]:
df.describe()

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,PutOuts,Assists,Errors,Salary
count,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.00000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,322.000000,263.000000
mean,380.928571,101.024845,10.770186,50.909938,48.027950,38.742236,7.444099,2648.68323,717.571429,69.490683,358.795031,330.118012,260.239130,288.937888,106.913043,8.040373,535.925882
std,153.404981,46.454741,8.709037,26.024095,26.166895,21.639327,4.926087,2324.20587,654.472627,86.266061,334.105886,333.219617,267.058085,280.704614,136.854876,6.368359,451.118681
min,16.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,19.00000,4.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,67.500000
25%,255.250000,64.000000,4.000000,30.250000,28.000000,22.000000,4.000000,816.75000,209.000000,14.000000,100.250000,88.750000,67.250000,109.250000,7.000000,3.000000,190.000000
50%,379.500000,96.000000,8.000000,48.000000,44.000000,35.000000,6.000000,1928.00000,508.000000,37.500000,247.000000,220.500000,170.500000,212.000000,39.500000,6.000000,425.000000
75%,512.000000,137.000000,16.000000,69.000000,64.750000,53.000000,11.000000,3924.25000,1059.250000,90.000000,526.250000,426.250000,339.250000,325.000000,166.000000,11.000000,750.000000
max,687.000000,238.000000,40.000000,130.000000,121.000000,105.000000,24.000000,14053.00000,4256.000000,548.000000,2165.000000,1659.000000,1566.000000,1378.000000,492.000000,32.000000,2460.000000


In [187]:
# fill null values and removing duplicates without errors
df.drop_duplicates(inplace=True, ignore_index=True)
for col in df.select_dtypes(include=[np.number]).columns:
    df[col].fillna(df[col].mean(), inplace=True)
        


C:\Users\ASUS\AppData\Local\Temp\ipykernel_10836\726675906.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)


In [188]:
# Create training and testing sets using an 80-20 split salary as target variable

X = df.drop('Salary', axis=1)
y = df['Salary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Reduce scale/skew of target: use log1p for stability
# (model trained on log(salary+1)); metrics are inverse-transformed for interpretability)
y_train = np.log1p(y_train)
y_test = np.log1p(y_test)

# Handle categorical variables using one-hot encoding
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)





In [189]:
from sklearn.preprocessing import StandardScaler

# Creating and training the linear regression model and scaling features


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)
# Making predictions (predictions are on log-scale)
y_pred = model.predict(X_test_scaled)
# Evaluating the model: report metrics on log-scale and original scale
mse_log = mean_squared_error(y_test, y_pred)
r2_log = r2_score(y_test, y_pred)
# inverse-transform back to original salary scale for interpretable metrics
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred)
# clip negative predictions (can't have negative salary)
y_pred_orig = np.maximum(y_pred_orig, 0)
y_test_orig = np.maximum(y_test_orig, 0)
mse_orig = mean_squared_error(y_test_orig, y_pred_orig)
rmse_orig = np.sqrt(mse_orig)
r2_orig = r2_score(y_test_orig, y_pred_orig)
print(f'Log-space MSE: {mse_log}')
print(f'Log-space R^2: {r2_log}')
print(f'Original-space MSE: {mse_orig}')
print(f'Original-space RMSE: {rmse_orig}')
print(f'Original-space R^2: {r2_orig}')
# Baseline comparison: predict mean(log-salary) and evaluate
baseline_log_pred = np.full_like(y_test, fill_value=y_train.mean())
print('Baseline log-space R^2:', r2_score(y_test, baseline_log_pred))
baseline_orig_pred = np.expm1(baseline_log_pred)
baseline_orig_pred = np.maximum(baseline_orig_pred, 0)
print('Baseline original-space R^2:', r2_score(y_test_orig, baseline_orig_pred))

Log-space MSE: 0.4656622757946631
Log-space R^2: 0.003610848449500881
Original-space MSE: 143334.82433324217
Original-space RMSE: 378.595858843229
Original-space R^2: 0.23870507959288467
Baseline log-space R^2: -0.05248006573605113
Baseline original-space R^2: -0.182553097881182


In [190]:
# Regularization techniques such as Ridge or Lasso regression can be implemented to improve model performance and prevent overfitting.

# Example: Using Ridge Regression
from sklearn.linear_model import Ridge
ridge_model = Ridge(alpha=1.0)
# Use scaled features (target is log-transformed)
ridge_model.fit(X_train_scaled, y_train)
ridge_y_pred = ridge_model.predict(X_test_scaled)
ridge_mse_log = mean_squared_error(y_test, ridge_y_pred)
ridge_r2_log = r2_score(y_test, ridge_y_pred)
ridge_y_pred_orig = np.expm1(ridge_y_pred)
ridge_y_test_orig = np.expm1(y_test)
ridge_y_pred_orig = np.maximum(ridge_y_pred_orig, 0)
ridge_y_test_orig = np.maximum(ridge_y_test_orig, 0)
ridge_mse_orig = mean_squared_error(ridge_y_test_orig, ridge_y_pred_orig)
ridge_rmse_orig = np.sqrt(ridge_mse_orig)
ridge_r2_orig = r2_score(ridge_y_test_orig, ridge_y_pred_orig)
print(f'Ridge Log-space MSE: {ridge_mse_log}')
print(f'Ridge Log-space R^2: {ridge_r2_log}')
print(f'Ridge Original-space MSE: {ridge_mse_orig}')
print(f'Ridge Original-space RMSE: {ridge_rmse_orig}')
print(f'Ridge Original-space R^2: {ridge_r2_orig}')
# If log-space R^2 is negative, try tuning alpha via GridSearchCV
from sklearn.model_selection import GridSearchCV
params = {'alpha':[0.01, 0.1, 1, 10, 100]}
gs = GridSearchCV(Ridge(), params, cv=5, scoring='r2')
gs.fit(X_train_scaled, y_train)
print('Ridge GridSearch best alpha:', gs.best_params_)
best_ridge = gs.best_estimator_
best_ridge_pred = best_ridge.predict(X_test_scaled)
print('Best Ridge log-space R^2:', r2_score(y_test, best_ridge_pred))


Ridge Log-space MSE: 0.4279721680129477
Ridge Log-space R^2: 0.08425730934303921
Ridge Original-space MSE: 140444.6029452054
Ridge Original-space RMSE: 374.7593934048957
Ridge Original-space R^2: 0.2540559259193075
Ridge GridSearch best alpha: {'alpha': 100}
Best Ridge log-space R^2: 0.34221720784898246


In [191]:
# Example: Using Lasso Regression
from sklearn.linear_model import Lasso
lasso_model = Lasso(alpha=0.1)
# Use scaled features (target is log-transformed)
lasso_model.fit(X_train_scaled, y_train)
lasso_y_pred = lasso_model.predict(X_test_scaled)
lasso_mse_log = mean_squared_error(y_test, lasso_y_pred)
lasso_r2_log = r2_score(y_test, lasso_y_pred)
lasso_y_pred_orig = np.expm1(lasso_y_pred)
lasso_y_test_orig = np.expm1(y_test)
lasso_y_pred_orig = np.maximum(lasso_y_pred_orig, 0)
lasso_y_test_orig = np.maximum(lasso_y_test_orig, 0)
lasso_mse_orig = mean_squared_error(lasso_y_test_orig, lasso_y_pred_orig)
lasso_rmse_orig = np.sqrt(lasso_mse_orig)
lasso_r2_orig = r2_score(lasso_y_test_orig, lasso_y_pred_orig)
print(f'Lasso Log-space MSE: {lasso_mse_log}')
print(f'Lasso Log-space R^2: {lasso_r2_log}')
print(f'Lasso Original-space MSE: {lasso_mse_orig}')
print(f'Lasso Original-space RMSE: {lasso_rmse_orig}')
print(f'Lasso Original-space R^2: {lasso_r2_orig}')
# Tune alpha for Lasso as well if needed
from sklearn.model_selection import GridSearchCV
params_l = {'alpha':[0.0001, 0.001, 0.01, 0.1, 1]}
gs_l = GridSearchCV(Lasso(max_iter=10000), params_l, cv=5, scoring='r2')
gs_l.fit(X_train_scaled, y_train)
print('Lasso GridSearch best alpha:', gs_l.best_params_)
best_lasso = gs_l.best_estimator_
print('Best Lasso log-space R^2:', r2_score(y_test, best_lasso.predict(X_test_scaled)))

Lasso Log-space MSE: 0.3354955411367896
Lasso Log-space R^2: 0.2821318475674286
Lasso Original-space MSE: 169171.00158304174
Lasso Original-space RMSE: 411.3040257316256
Lasso Original-space R^2: 0.10148127097201864
Lasso GridSearch best alpha: {'alpha': 0.1}
Best Lasso log-space R^2: 0.2821318475674286


In [192]:
# Target scaling applied: salaries are log1p-transformed before training.
# Metrics printed now include both log-space and original-space (inverse-transformed) values.
# Run the notebook to see updated MSE/R^2 outputs.

In [193]:
# Create random forest model as another approach to compare results
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=150, random_state=42)
# Use original features (not scaled), target is log-transformed
rf_model.fit(X_train, y_train)
rf_y_pred = rf_model.predict(X_test)
rf_mse_log = mean_squared_error(y_test, rf_y_pred)
rf_r2_log = r2_score(y_test, rf_y_pred)
rf_y_pred_orig = np.expm1(rf_y_pred)
rf_y_test_orig = np.expm1(y_test)
rf_y_pred_orig = np.maximum(rf_y_pred_orig, 0)
rf_y_test_orig = np.maximum(rf_y_test_orig, 0)
rf_mse_orig = mean_squared_error(rf_y_test_orig, rf_y_pred_orig)
rf_rmse_orig = np.sqrt(rf_mse_orig)
rf_r2_orig = r2_score(rf_y_test_orig, rf_y_pred_orig)
print(f'Random Forest Log-space MSE: {rf_mse_log}')
print(f'Random Forest Log-space R^2: {rf_r2_log}')
print(f'Random Forest Original-space MSE: {rf_mse_orig}')
print(f'Random Forest Original-space RMSE: {rf_rmse_orig}')
print(f'Random Forest Original-space R^2: {rf_r2_orig}')


Random Forest Log-space MSE: 0.1628254451943195
Random Forest Log-space R^2: 0.65159834579441
Random Forest Original-space MSE: 99524.68984965696
Random Forest Original-space RMSE: 315.47533952696995
Random Forest Original-space R^2: 0.4713940510264021
